### HealthyAir - Data Cleaning Notebook
#### Examines the dataset structure, data types, missing values, distributions, 
#### correlations and time patterns. A fixed seed makes resampling repeatable.
#### Inspects semantic variable roles, diagnoses missingness, compares groups, constructs a bootstrap interval, examines conditional patterns, distribution-shift and leakage risks.

#### Expected input: data/raw/AirQualityUCI.csv
#### The original UCI file is commonly semicolon-separated, uses decimal commas, and 
#### encodes missing readings as -200. 
#### Check downloaded copy and keep the raw file unchanged.

In [1]:
# Import Libraries
import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import statsmodels
import scipy
import os
import gcsfs
import warnings
from scipy import stats

# Confirm location
print("Location:", sys.executable)

SEED = 42
rng = np.random.default_rng(SEED)
#pd.set_option("display.max_columns", 30)
#pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", sns.__version__)
print("statsmodels:", statsmodels.__version__)

Location: /home/chloekc/datahandling/healthyair/.venv/bin/python
Python: 3.12.3
NumPy: 2.5.3
pandas: 3.0.6
SciPy: 1.18.1
Matplotlib: 3.11.1
Seaborn: 0.13.2
statsmodels: 0.15.0


In [2]:
# Load data from Google Cloud Storage
GCS_PROJECT_ID = "128630137158"
GCS_BUCKET = "uci_air_quality_20119102"
GCS_FILE = "gs://uci_air_quality_20119102/data_raw/AirQualityUCI.csv"
df = pd.read_csv(GCS_FILE)

# Data contract
required = {"Date", "Time", "CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)",
            "PT08.S2(NMHC)", "NOx(GT)", "PT08.S3(NOx)", "NO2(GT)", "PT08.S4(NO2)", "PT08.S5(O3)", "T", "RH", "AH"}
assert required.issubset(df.columns)
#assert df["PassengerId"].is_unique
#assert set(df["Quality"].unique()).issubset({0, 1, 3})
print("Initial data-contract checks passed.")

print("Dataset path:", GCS_BUCKET)
print(f"Rows: {df.shape[0]:,}; columns: {df.shape[1]}")
display(df.head())

Rows: 9,471; columns: 17


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,3/10/2004,18:00:00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,NaN,NaN
1,3/10/2004,19:00:00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,NaN,NaN
2,3/10/2004,20:00:00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,NaN,NaN
3,3/10/2004,21:00:00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,NaN,NaN
4,3/10/2004,22:00:00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,NaN,NaN


#### Timestamp Analysis

In [ ]:
print("Date range:")
print(df["Date"].min(), "to", df["Date"].max())
display(df[["Date", "Time"]].head(10))

In [ ]:
# Combine Date and Time into one datetime column

df["Datetime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    format="%d/%m/%Y %H.%M.%S",
    errors="coerce"
)

# Check the results
print(df["Datetime"].head())
print("\nMissing timestamps:", df["Datetime"].isna().sum())

print("\nFirst timestamp:", df["Datetime"].min())
print("Last timestamp:", df["Datetime"].max())

In [ ]:
# Sort chronologically
df = df.sort_values("Datetime").reset_index(drop=True)

# Calculate time differences between observations
df["Time_Difference"] = df["Datetime"].diff()

# Check the intervals
print(df["Time_Difference"].value_counts().head(10))

# Identify gaps larger than one hour
gaps = df[df["Time_Difference"] > pd.Timedelta(hours=1)]

print("\nGaps larger than one hour:", len(gaps))
display(gaps[["Datetime", "Time_Difference"]].head(20))